# USDA Phytochemical & Ethnobotanical Database — Enriched v2.3

**76,907 records · 24,746 compounds · 2,313 species · 5 enrichment layers**

| Tier | Price | Includes |
|------|-------|----------|
| **Single Entity** | [€699](https://buy.stripe.com/00w6oGgFh58v6Toeqsebu02?utm_source=github&utm_medium=notebook&utm_campaign=launch_2026_03) | JSON + Parquet + SHA-256 Manifest |
| **Team** | [€1,349](https://buy.stripe.com/dRm7sK9cP1Wj0v06Y0ebu03?utm_source=github&utm_medium=notebook&utm_campaign=launch_2026_03) | + `duckdb_queries.sql` (20 queries) + `compound_priority_score.py` + 4 Pre-computed Views |
| **Enterprise** | [€1,699](mailto:founder@ethno-api.com?subject=Enterprise%20License%20Inquiry) | + `snowflake_load.sql` + `chromadb_ingest.py` + `pinecone_ingest.py` + `embedding_guide.md` + Opportunity Matrix |

**Full dataset:** [ethno-api.com](https://ethno-api.com) · **This notebook runs on the free 400-row sample.**


In [ ]:
import pandas as pd
import duckdb

# Load the free 400-row sample
# To use full dataset: replace path with 'ethno_dataset_2026_v2.3.json'
SAMPLE = 'ethno_sample_400.json'

df = pd.read_json(SAMPLE)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# Dataset statistics
print(f'Unique compounds:  {df["chemical"].nunique():,}')
print(f'Unique species:    {df["plant_species"].nunique():,}')
print(f'Application coverage: {df["application"].notna().mean()*100:.1f}%')
print(f'Dosage coverage:      {df["dosage"].notna().mean()*100:.1f}%')
print(f'\nTop compound by PubMed evidence:')
print(df.groupby("chemical")["pubmed_mentions_2026"].max().sort_values(ascending=False).head(5).to_string())

In [ ]:
# Q05: Composite evidence score (weighted: PubMed 30% | Trials 35% | ChEMBL 20% | Patents 15%)
# Source: duckdb_queries.sql — available in Team + Enterprise tier
result = duckdb.sql(f"""
    SELECT
        chemical,
        MAX(pubmed_mentions_2026)       AS pubmed,
        MAX(clinical_trials_count_2026) AS trials,
        MAX(chembl_bioactivity_count)   AS bioassays,
        MAX(patent_count_since_2020)    AS patents,
        ROUND(
            (MAX(pubmed_mentions_2026)       * 0.30) +
            (MAX(clinical_trials_count_2026) * 0.35) +
            (MAX(chembl_bioactivity_count)   * 0.20) +
            (MAX(patent_count_since_2020)    * 0.15),
        2) AS composite_score
    FROM read_json_auto('{SAMPLE}')
    GROUP BY chemical
    ORDER BY composite_score DESC
    LIMIT 10
""").df()
print('Top 10 by composite evidence score:')
result

In [ ]:
# Q17: Patent-Literature Gap candidates — high research signal, low patent activity
# Use case: IP-Literature Discrepancy screening for drug discovery pipeline gaps
# Source: duckdb_queries.sql (Team + Enterprise tier)
gap_candidates = duckdb.sql(f"""
    SELECT
        chemical,
        MAX(pubmed_mentions_2026)    AS pubmed,
        MAX(patent_count_since_2020) AS patents_since_2020,
        ROUND(MAX(pubmed_mentions_2026)::DOUBLE / NULLIF(MAX(patent_count_since_2020),0), 1)
            AS research_to_patent_ratio
    FROM read_json_auto('{SAMPLE}')
    GROUP BY chemical
    HAVING MAX(pubmed_mentions_2026) > 50 AND MAX(patent_count_since_2020) < 5
    ORDER BY research_to_patent_ratio DESC
    LIMIT 10
""").df()
print('Patent-Literature Gap Candidates (high research, low patents):')
gap_candidates

In [ ]:
# Q06: Evidence tier classification (PLATINUM / GOLD / SILVER / BRONZE)
# Source: duckdb_queries.sql (Team + Enterprise tier)
tiers = duckdb.sql(f"""
    SELECT evidence_tier, COUNT(*) AS compound_count
    FROM (
        SELECT chemical,
            CASE
                WHEN MAX(pubmed_mentions_2026)>5000 AND MAX(clinical_trials_count_2026)>100 THEN 'PLATINUM'
                WHEN MAX(pubmed_mentions_2026)>1000 AND MAX(clinical_trials_count_2026)>20  THEN 'GOLD'
                WHEN MAX(pubmed_mentions_2026)>100 THEN 'SILVER'
                ELSE 'BRONZE'
            END AS evidence_tier
        FROM read_json_auto('{SAMPLE}')
        GROUP BY chemical
    ) sub
    GROUP BY evidence_tier
    ORDER BY evidence_tier
""").df()
print('Evidence tier distribution:')
tiers

## What's Included in Each Tier

### Team License (€1,349, reg. €1.349) — adds to Single:
| File | Description |
|------|-------------|
| `duckdb_queries.sql` | 20 production-ready queries across 5 categories (compound discovery, evidence scoring, species analysis, application intelligence, IP/pipeline) |
| `compound_priority_score.py` | Combined evidence score calculator across all 4 layers |
| `top500_by_pubmed.parquet` | Pre-computed Top 500 compounds by PubMed score |
| `top500_by_trials.parquet` | Pre-computed Top 500 by ClinicalTrials count |
| `top500_by_patent_density.parquet` | Pre-computed Top 500 by patent density |
| `anti_inflammatory_panel.parquet` | All anti-inflammatory application records |

### Enterprise License (€1,699, reg. €1.699) — adds to Team:
| File | Description |
|------|-------------|
| `snowflake_load.sql` | Drop-in Snowflake import script (Stage + COPY INTO + Verify) |
| `chromadb_ingest.py` | ChromaDB vector ingest with batch upload, --resume, verification |
| `pinecone_ingest.py` | Pinecone ingest supporting OpenAI, sentence-transformers, Pinecone inference |
| `embedding_guide.md` | ClinicalBERT, RAG pipeline templates, cost benchmarks |
| `compound_opportunity_matrix.csv` | Ranked compound candidates: high bioactivity × low patent density |
| `clinical_pipeline_gaps.csv` | High-PubMed, low-trial compounds: drug discovery pipeline gaps |
| `ethno_rag_chunks.jsonl` | Pre-chunked JSONL for direct LLM embedding (ChromaDB / Pinecone) |

---

**→ Purchase the full dataset:**

[**Single €699 →**](https://buy.stripe.com/00w6oGgFh58v6Toeqsebu02?utm_source=github&utm_medium=notebook&utm_campaign=launch_2026_03) &nbsp;&nbsp;[**Team €1,349 →**](https://buy.stripe.com/dRm7sK9cP1Wj0v06Y0ebu03?utm_source=github&utm_medium=notebook&utm_campaign=launch_2026_03) &nbsp;&nbsp;[**Enterprise €1,699 →**](mailto:founder@ethno-api.com?subject=Enterprise%20License%20Inquiry)


In [ ]:
# Load via HuggingFace Datasets
# from datasets import load_dataset
# ds = load_dataset(
#     'wirthal1990-tech/USDA-Phytochemical-Database-JSON',
#     split='sample',
#     trust_remote_code=False
# )
# df_hf = ds.to_pandas()
# print(f'Records: {len(df_hf)} | Columns: {list(df_hf.columns)}')
print('HuggingFace repo: https://huggingface.co/datasets/wirthal1990-tech/USDA-Phytochemical-Database-JSON')

## Citation

```bibtex
@misc{ethno_api_v2.3_2026,
  title     = {USDA Phytochemical \& Ethnobotanical Database --- Enriched v2.3},
  author    = {Wirth, Alexander},
  year      = {2026},
  publisher = {Ethno-API},
  url       = {https://ethno-api.com},
  doi       = {10.5281/zenodo.19265853},
  note      = {76,907 records, 24,746 unique chemicals, 2,313 plant species,
               10-column schema with PubMed, ClinicalTrials, ChEMBL, and PatentsView enrichment}
}
```

[![DOI](https://zenodo.org/badge/doi/10.5281/zenodo.19265853.svg)](https://zenodo.org/records/19265853)

**Links:** [ethno-api.com](https://ethno-api.com) · [GitHub](https://github.com/wirthal1990-tech/USDA-Phytochemical-Database-JSON) · [HuggingFace](https://huggingface.co/datasets/wirthal1990-tech/USDA-Phytochemical-Database-JSON)


---

## Purchase Full Dataset

| Tier | Price | |
|------|-------|---|
| Single Entity | €699 netto | [**Buy →**](https://buy.stripe.com/00w6oGgFh58v6Toeqsebu02?utm_source=github&utm_medium=notebook&utm_campaign=launch_2026_03) |
| Team | €1,349 netto | [**Buy →**](https://buy.stripe.com/dRm7sK9cP1Wj0v06Y0ebu03?utm_source=github&utm_medium=notebook&utm_campaign=launch_2026_03) |
| Enterprise | €1,699 netto | [**Contact →**](mailto:founder@ethno-api.com) |

> Gemäß § 19 UStG keine Umsatzsteuer.